
# OPTIMAL WEEKLY ROUTING WITH GOOGLE OR-TOOLS
Constraints:
- 20 employees working over 5 days (Monday-Friday)
- Max 3 clients per employee per day
- Max 6 hours work time per day (travel + care hours)
- Compatibility based on pets (dogs/cats) and smoking
- Interactive Map with day selection


# 1. Import Libraries

In [19]:
import pandas as pd
import numpy as np
import networkx as nx
from shapely import wkt
import folium
from scipy.spatial import cKDTree
from ortools.constraint_solver import routing_enums_pb2, pywrapcp
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime, timedelta
from ipywidgets import interact, Dropdown
from IPython.display import IFrame, display, HTML

print('Libraries geladen.')

Libraries geladen.


# 2. Wegennet laden en graph bouwen

In [20]:
edges_df = pd.read_csv('../output/heerlen_edge_table_traveltypes.csv')
print(f'Aantal edges: {len(edges_df)}')
edges_df['geometry'] = edges_df['geometry'].apply(wkt.loads)

# Mapping van medewerker voertuigtype naar edge transportation_type
VEHICLE_TO_TRANSPORT = {'car': 'car', 'walking': 'pedestrian', 'bike': 'bike'}
TRANSPORT_TYPES = ['car', 'pedestrian', 'bike']

# Bouw een aparte NetworkX graph per transporttype
graphs = {}
node_coords = {}
edge_geom_all = {}  # (type, u, v) -> geom

for transport in TRANSPORT_TYPES:
    sub = edges_df[edges_df['transportation_type'] == transport]
    G_sub = nx.Graph()
    for _, row in sub.iterrows():
        geom = row['geometry']
        coords = list(geom.coords)
        u, v = row['u'], row['v']
        G_sub.add_edge(u, v, weight=row['travel_time_min'], geometry=geom)
        node_coords[u] = (coords[0][0], coords[0][1])
        node_coords[v] = (coords[-1][0], coords[-1][1])
        edge_geom_all[(transport, u, v)] = geom
        edge_geom_all[(transport, v, u)] = geom
    graphs[transport] = G_sub
    print(f'Graph [{transport}]: {G_sub.number_of_nodes()} knopen, {G_sub.number_of_edges()} takken.')

# KD-tree over alle unieke knopen (voor nearest-node lookup)
node_ids = list(node_coords.keys())
node_lons_arr = np.array([node_coords[n][0] for n in node_ids])
node_lats_arr = np.array([node_coords[n][1] for n in node_ids])
kd_tree = cKDTree(np.column_stack((node_lons_arr, node_lats_arr)))

def nearest_node(lon, lat):
    _, idx = kd_tree.query([lon, lat])
    return node_ids[idx]


Aantal edges: 21096
Graph [car]: 3109 knopen, 4311 takken.
Graph [pedestrian]: 3015 knopen, 4146 takken.
Graph [bike]: 3015 knopen, 4146 takken.


# 3. Medewerkers laden (inclusief huisdier- en rookvoorkeuren)

In [21]:
emp_csv = pd.read_csv('../output/employees_vehicle_type.csv')
emp_csv['name'] = emp_csv['name'].str.strip()

def split_coords_emp(s):
    parts = str(s).replace(';', ' ').replace(',', ' ').split()
    return (float(parts[0]), float(parts[1])) if len(parts) == 2 else (np.nan, np.nan)

emp_csv[['lat', 'lon']] = emp_csv['coordinates'].apply(lambda x: pd.Series(split_coords_emp(x)))
emp_csv = emp_csv.dropna(subset=['lat', 'lon']).reset_index(drop=True)
emp_csv['node'] = emp_csv.apply(lambda r: nearest_node(r['lon'], r['lat']), axis=1)
emp_csv['transport_type'] = emp_csv['vehicle_type'].map(VEHICLE_TO_TRANSPORT)

employees_df = emp_csv[['name', 'lat', 'lon', 'node', 'dogs', 'cats', 'smokes', 'vehicle_type', 'transport_type']].copy()
employees_df['dogs'] = employees_df['dogs'].fillna(-1).astype(int)
employees_df['cats'] = employees_df['cats'].fillna(-1).astype(int)
employees_df['smokes'] = employees_df['smokes'].fillna(False).astype(bool)

print(f'Aantal medewerkers: {len(employees_df)}')
print(employees_df[['name', 'vehicle_type', 'transport_type']].to_string())


Aantal medewerkers: 20
           name vehicle_type transport_type
0    employee 1          car            car
1    employee 2      walking     pedestrian
2    employee 3         bike           bike
3    employee 4          car            car
4    employee 5      walking     pedestrian
5    employee 6         bike           bike
6    employee 7          car            car
7    employee 8      walking     pedestrian
8    employee 9         bike           bike
9   employee 10          car            car
10  employee 11      walking     pedestrian
11  employee 12         bike           bike
12  employee 13          car            car
13  employee 14      walking     pedestrian
14  employee 15         bike           bike
15  employee 16          car            car
16  employee 17      walking     pedestrian
17  employee 18         bike           bike
18  employee 19          car            car
19  employee 20      walking     pedestrian


# 3b hulpfuncties voor afstandsberekening

In [22]:
# Cel 3b – Haversine afstand en lengte van een LineString in km
from math import radians, sin, cos, sqrt, atan2

def haversine(lon1, lat1, lon2, lat2):
    """Bereken de afstand in km tussen twee (lon,lat) punten."""
    R = 6371.0  # Aardstraal in km
    dlon = radians(lon2 - lon1)
    dlat = radians(lat2 - lat1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c

def line_length_km(geom):
    """Bereken de lengte van een Shapely LineString in km via haversine."""
    if geom is None or geom.is_empty:
        return 0.0
    coords = list(geom.coords)
    total = 0.0
    for i in range(len(coords)-1):
        lon1, lat1 = coords[i]
        lon2, lat2 = coords[i+1]
        total += haversine(lon1, lat1, lon2, lat2)
    return total

# 4. Cliënten laden (coördinaten, zorgtijd, huisdieren, rook, tijdvenster)

In [23]:
clients_df = pd.read_csv('../output/clients.csv')

# Coördinaten detecteren
coord_col = None
for col in clients_df.columns:
    sample = clients_df[col].dropna().astype(str).iloc[0]
    parts = sample.replace(',', ' ').replace(';', ' ').split()
    if len(parts) == 2:
        try:
            float(parts[0]); float(parts[1])
            coord_col = col
            break
        except:
            pass
coord_col = coord_col or clients_df.columns[0]

def split_coords(s):
    parts = str(s).replace(';', ' ').replace(',', ' ').split()
    return (float(parts[0]), float(parts[1])) if len(parts) == 2 else (np.nan, np.nan)

clients_df[['lat', 'lon']] = clients_df[coord_col].apply(lambda x: pd.Series(split_coords(x)))
clients_df = clients_df.dropna(subset=['lat', 'lon']).reset_index(drop=True)
clients_df['client_id'] = clients_df.index
clients_df['node'] = clients_df.apply(lambda r: nearest_node(r['lon'], r['lat']), axis=1)

# Kolommen standaard invullen
for col in ['dogs', 'cats', 'smokes', 'care_hours']:
    if col not in clients_df.columns:
        clients_df[col] = 0 if col in ['dogs', 'cats'] else (False if col == 'smokes' else 1.0)
clients_df['smokes'] = clients_df['smokes'].astype(bool)
clients_df['care_hours'] = pd.to_numeric(clients_df['care_hours'], errors='coerce').fillna(1.0)

# Tijdvenster omzetten naar minuten na 07:00
def window_to_minutes(t_str):
    h, m = map(int, t_str.split(':'))
    return h * 60 + m - 420  # 07:00 = 0
clients_df['tw_min'] = clients_df['time_window_start'].apply(window_to_minutes)
clients_df['tw_max'] = clients_df['time_window_end'].apply(window_to_minutes)

print(f'Aantal cliënten: {len(clients_df)}')

Aantal cliënten: 100


# 5. Reistijdenmatrix berekenen

In [ ]:
# Cel 7 – Reistijdenmatrix berekenen (met minimum 5 min reistijd)
N_EMPLOYEES = len(employees_df)
N_CLIENTS = len(clients_df)
N_TOTAL = N_EMPLOYEES + N_CLIENTS
SCALE = 100

# Groepeer employees per transporttype
transport_groups = employees_df.groupby('transport_type').groups  # {type: [emp_id, ...]}

all_nodes = employees_df['node'].tolist() + clients_df['node'].tolist()

print(f'Dijkstra uitvoeren per transporttype ...')

# Sla per transport type de shortest paths op vanuit alle relevante knopen
dist_from = {t: {} for t in TRANSPORT_TYPES}
for transport, G_t in graphs.items():
    emp_ids_for_type = transport_groups.get(transport, [])
    sources = set()
    for eid in emp_ids_for_type:
        sources.add(employees_df.iloc[eid]['node'])
    for cid in range(N_CLIENTS):
        sources.add(clients_df.iloc[cid]['node'])
    
    for i, src in enumerate(sources):
        if src not in G_t:
            dist_from[transport][src] = {}
            continue
        dist_from[transport][src] = nx.single_source_dijkstra_path_length(G_t, src, weight='weight')
    print(f'  [{transport}] {len(sources)} bronknopen berekend.')

time_matrices = {}
MIN_TRAVEL_SCALED = 5 * SCALE  # 5 minuten minimum

for transport in TRANSPORT_TYPES:
    mat = np.zeros((N_TOTAL, N_TOTAL), dtype=np.int64)
    df_t = dist_from[transport]
    for i in range(N_TOTAL):
        src_node = all_nodes[i]
        lengths = df_t.get(src_node, {})
        for j in range(N_TOTAL):
            dst_node = all_nodes[j]
            t = lengths.get(dst_node, float('inf'))
            mat[i][j] = int(t * SCALE) if t != float('inf') else 10_000_000
    # Zet minimale reistijd op 5 minuten voor alle niet-diagonale waarden
    np.fill_diagonal(mat, 0)
    mat[mat > 0] = np.maximum(mat[mat > 0], MIN_TRAVEL_SCALED)
    time_matrices[transport] = mat
    print(f'[{transport}] matrix klaar. Min: {mat[mat>0].min()/SCALE:.1f} min, Max (reachable): {mat[mat<10_000_000].max()/SCALE:.1f} min')

emp_transport_list = employees_df['transport_type'].tolist()
print(f'Reistijdmatrices klaar per transporttype (minimum reistijd = 5 min).')

Dijkstra uitvoeren per transporttype ...
  [car] 89 bronknopen berekend.
  [pedestrian] 89 bronknopen berekend.
  [bike] 89 bronknopen berekend.
[car] matrix klaar. Min: 0.0 min, Max (reachable): 11.5 min
[pedestrian] matrix klaar. Min: 0.4 min, Max (reachable): 135.4 min
[bike] matrix klaar. Min: 0.1 min, Max (reachable): 45.1 min
Reistijdmatrices klaar per transporttype.


# 6. Meerdaags model instellen (100 voertuigen: 20 medewerkers × 5 dagen)

In [25]:
NUM_DAYS = 5
MAX_CLIENTS_PER_VEHICLE = 3
MAX_WORK_MINUTES = 360  # 6 uur

vehicles = []
for emp_id in range(N_EMPLOYEES):
    for day in range(NUM_DAYS):
        vehicles.append({
            'vehicle_id': len(vehicles),
            'emp_id': emp_id,
            'day': day,
            'start_node': emp_id,
            'end_node': emp_id
        })
N_VEHICLES = len(vehicles)
print(f'Aantal voertuigen (medewerker×dag): {N_VEHICLES}')

Aantal voertuigen (medewerker×dag): 100


# 7. OR-Tools data model (tijdvensters, capaciteit, werktijdlimiet)

In [26]:
data = {}
data['num_vehicles'] = N_VEHICLES
data['starts'] = [v['start_node'] for v in vehicles]
data['ends']   = [v['end_node'] for v in vehicles]
data['demands'] = [0] * N_EMPLOYEES + [1] * N_CLIENTS
data['capacities'] = [MAX_CLIENTS_PER_VEHICLE] * N_VEHICLES

service_time = [0] * N_EMPLOYEES + (clients_df['care_hours'] * 60).round().astype(int).tolist()

time_windows = [(0, 660)] * N_EMPLOYEES
for _, client in clients_df.iterrows():
    time_windows.append((client['tw_min'], client['tw_max']))

manager = pywrapcp.RoutingIndexManager(
    N_TOTAL,
    data['num_vehicles'],
    data['starts'],
    data['ends']
)
routing = pywrapcp.RoutingModel(manager)

vehicle_transport = [emp_transport_list[v['emp_id']] for v in vehicles]

# Transit callbacks per transporttype (reistijd + zorgtijd)
def make_transit_callback(transport_type):
    mat = time_matrices[transport_type]
    def callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        travel = int(mat[from_node][to_node])
        service = service_time[from_node] * SCALE
        return travel + service
    return callback

transit_callbacks = {}
for transport in TRANSPORT_TYPES:
    cb = make_transit_callback(transport)
    transit_callbacks[transport] = routing.RegisterTransitCallback(cb)

# Kostoptimalisatie: elk voertuig gebruikt zijn eigen reistijden
for v_id in range(N_VEHICLES):
    transport = vehicle_transport[v_id]
    routing.SetArcCostEvaluatorOfVehicle(transit_callbacks[transport], v_id)

# Capaciteitsdimensie
def demand_callback(from_index):
    from_node = manager.IndexToNode(from_index)
    return data['demands'][from_node]

demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index, 0, data['capacities'], True, 'Capacity'
)

# ✅ SLIMME FIX: tijdsdimensie per voertuig met zijn eigen reistijden
# Bouw een lijst van callback indices, één per voertuig
vehicle_transit_callbacks = [
    transit_callbacks[vehicle_transport[v]] for v in range(N_VEHICLES)
]

routing.AddDimensionWithVehicleTransits(
    vehicle_transit_callbacks,
    30 * SCALE,       # wacht-slack: max 30 min vroeger aankomen
    720 * SCALE,      # max cumulatieve tijd (ruim; beperken via Span)
    False,
    'Time'
)
time_dim = routing.GetDimensionOrDie('Time')

# Tijdvensters per node
for node in range(N_TOTAL):
    index = manager.NodeToIndex(node)
    tw_min, tw_max = time_windows[node]
    time_dim.CumulVar(index).SetRange(tw_min * SCALE, tw_max * SCALE)

# Max werktijd per voertuig (6 uur) — nu correct per transporttype!
MAX_WORK_SCALED = MAX_WORK_MINUTES * SCALE
for v in range(N_VEHICLES):
    time_dim.SetSpanUpperBoundForVehicle(MAX_WORK_SCALED, v)

print('Data model klaar. Tijdsdimensie nu per voertuig correct berekend.')
print('Voetgangers/fietsers hebben minder reisbereikte clients door hogere reistijden.')

Data model klaar. Tijdsdimensie nu per voertuig correct berekend.
Voetgangers/fietsers hebben minder reisbereikte clients door hogere reistijden.


# 8. Compatibiliteit (huisdieren/rook) en voertuigbeperkingen

In [27]:
compatible_emp_per_client = []
skipped_clients = []
for cid in range(N_CLIENTS):
    client = clients_df.iloc[cid]
    compatible = []
    for emp_id, emp in employees_df.iterrows():
        if emp['dogs'] != -1 and client['dogs'] > emp['dogs']:
            continue
        if emp['cats'] != -1 and client['cats'] > emp['cats']:
            continue
        if not emp['smokes'] and client['smokes']:
            continue
        compatible.append(emp_id)
    if not compatible:
        skipped_clients.append(cid)
    compatible_emp_per_client.append(compatible)

solver = routing.solver()
for cid in range(N_CLIENTS):
    node_idx = manager.NodeToIndex(N_EMPLOYEES + cid)
    if cid in skipped_clients:
        routing.AddDisjunction([node_idx], 10_000_000)
    else:
        vehicle_var = routing.VehicleVar(node_idx)
        for v in range(N_VEHICLES):
            emp_id = vehicles[v]['emp_id']
            if emp_id not in compatible_emp_per_client[cid]:
                solver.Add(vehicle_var != v)

print(f'Cliënten zonder geschikte medewerker: {len(skipped_clients)}')

Cliënten zonder geschikte medewerker: 0


# 9. VRP oplossen

In [28]:
search_params = pywrapcp.DefaultRoutingSearchParameters()
search_params.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
)
search_params.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_params.time_limit.seconds = 180
search_params.log_search = True

print('Meerdaagse VRP wordt opgelost (max 3 cliënten/dag, max 6u/dag, tijdvensters)...')
solution = routing.SolveWithParameters(search_params)

if solution:
    print(f'\nOplossing gevonden! Totale reistijd: {solution.ObjectiveValue() / SCALE:.1f} min')
else:
    print('\nGeen oplossing gevonden.')

Meerdaagse VRP wordt opgelost (max 3 cliënten/dag, max 6u/dag, tijdvensters)...

Oplossing gevonden! Totale reistijd: 12740.8 min


# 10. Routes extraheren

In [ ]:
def extract_routes_safe(solution, routing, manager):
    routes = []
    time_dim = routing.GetDimensionOrDie('Time')
    
    for v in range(N_VEHICLES):
        try:
            if routing.IsVehicleUsed(solution, v) == False:
                continue
                
            index = routing.Start(v)
            nodes = []
            while not routing.IsEnd(index):
                node = manager.IndexToNode(index)
                if node < 0 or node >= N_TOTAL:
                    raise ValueError(f"Ongeldige node index {node} bij voertuig {v}")
                nodes.append(node)
                index = solution.Value(routing.NextVar(index))
            end_node = manager.IndexToNode(index)
            if end_node < 0 or end_node >= N_TOTAL:
                raise ValueError(f"Ongeldige eindnode index {end_node} bij voertuig {v}")
            nodes.append(end_node)

            client_ids = [n - N_EMPLOYEES for n in nodes if n >= N_EMPLOYEES]
            if not client_ids:
                continue

            start_cumul = solution.Value(time_dim.CumulVar(routing.Start(v)))
            end_cumul = solution.Value(time_dim.CumulVar(routing.End(v)))
            work_time = (end_cumul - start_cumul) / SCALE

            emp_id = vehicles[v]['emp_id']
            transport = emp_transport_list[emp_id]
            mat = time_matrices[transport]

            travel_time = 0.0
            travel_distance = 0.0

            for i in range(len(nodes)-1):
                node_from = nodes[i]
                node_to = nodes[i+1]
                if node_from >= N_TOTAL or node_to >= N_TOTAL:
                    raise ValueError(f"Segment ({node_from}->{node_to}) buiten matrix bij voertuig {v}")
                travel_time += mat[node_from][node_to] / SCALE

                u = all_nodes[node_from]
                v_node = all_nodes[node_to]
                geom = edge_geom_all.get((transport, u, v_node))
                if geom:
                    travel_distance += line_length_km(geom)
                else:
                    coord_u = node_coords.get(u)
                    coord_v = node_coords.get(v_node)
                    if coord_u and coord_v:
                        travel_distance += haversine(coord_u[0], coord_u[1], coord_v[0], coord_v[1])

            # Reiskosten (alleen voor auto)
            travel_cost = 0.0
            if transport == 'car':
                travel_cost = travel_distance * 0.23

            routes.append({
                'vehicle_id': v,
                'emp_id': emp_id,
                'day': vehicles[v]['day'],
                'transport_type': transport,
                'nodes': nodes,
                'client_ids': client_ids,
                'work_time': work_time,
                'travel_time': travel_time,
                'travel_distance': travel_distance,
                'travel_cost': travel_cost
            })

        except Exception as e:
            print(f"Waarschuwing: route voor voertuig {v} (emp {vehicles[v]['emp_id']}, dag {vehicles[v]['day']}) "
                  f"kon niet worden uitgelezen: {e}")
            continue

    return routes

56 routes geëxtraheerd (alleen routes met minimaal 1 cliënt).


# 11. Overzicht van niet-ingeplande cliënten

In [ ]:
# Cel 14 – Samenvatting per medewerker per dag (met kosten)
if solution:
    print('\n=== Samenvatting per medewerker per dag ===')
    for day in range(NUM_DAYS):
        dag_str = ['maandag', 'dinsdag', 'woensdag', 'donderdag', 'vrijdag'][day]
        print(f'\n--- Dag {day+1} ({dag_str}) ---')
        dag_routes = [r for r in all_routes if r['day'] == day and r['client_ids']]
        for r in dag_routes:
            emp_name = employees_df.loc[r['emp_id'], 'name']
            cost_str = f" | kosten €{r['travel_cost']:.2f}" if r['transport_type'] == 'car' else ""
            print(f'{emp_name:15s}: {len(r["client_ids"])} cliënten | werktijd {r["work_time"]:.1f} min | reistijd {r["travel_time"]:.1f} min | afstand {r["travel_distance"]:.2f} km{cost_str} | stops: {r["client_ids"]}')


Alle cliënten zijn ingepland!


# 12. Samenvatting per medewerker per dag

In [36]:
# Cel 14 – Samenvatting per medewerker per dag
if solution:
    print('\n=== Samenvatting per medewerker per dag ===')
    for day in range(NUM_DAYS):
        dag_str = ['maandag', 'dinsdag', 'woensdag', 'donderdag', 'vrijdag'][day]
        print(f'\n--- Dag {day+1} ({dag_str}) ---')
        dag_routes = [r for r in all_routes if r['day'] == day and r['client_ids']]
        for r in dag_routes:
            emp_name = employees_df.loc[r['emp_id'], 'name']
            print(f'{emp_name:15s}: {len(r["client_ids"])} cliënten | werktijd {r["work_time"]:.1f} min | reistijd {r["travel_time"]:.1f} min | afstand {r["travel_distance"]:.2f} km | stops: {r["client_ids"]}')


=== Samenvatting per medewerker per dag ===

--- Dag 1 (maandag) ---
employee 1     : 2 cliënten | werktijd 276.4 min | reistijd 6.4 min | afstand 3.94 km | stops: [95, 98]
employee 3     : 2 cliënten | werktijd 246.5 min | reistijd 6.5 min | afstand 0.88 km | stops: [3, 6]
employee 4     : 2 cliënten | werktijd 250.6 min | reistijd 10.6 min | afstand 5.81 km | stops: [43, 80]
employee 5     : 1 cliënten | werktijd 180.0 min | reistijd 0.0 min | afstand 0.00 km | stops: [65]
employee 6     : 2 cliënten | werktijd 242.4 min | reistijd 2.4 min | afstand 0.58 km | stops: [12, 50]
employee 7     : 1 cliënten | werktijd 150.0 min | reistijd 0.0 min | afstand 0.00 km | stops: [24]
employee 8     : 1 cliënten | werktijd 120.0 min | reistijd 0.0 min | afstand 0.00 km | stops: [77]
employee 9     : 2 cliënten | werktijd 158.3 min | reistijd 8.3 min | afstand 1.44 km | stops: [78, 38]
employee 10    : 1 cliënten | werktijd 180.0 min | reistijd 0.0 min | afstand 0.00 km | stops: [1]
employee 11 

# 13. Interactieve kaart (dropdown voor dagen)

In [37]:
# Cel 15 – Interactieve kaart (dropdown voor dagen)
if solution:
    EMPLOYEE_COLORS = [
        '#e6194b','#3cb44b','#ffe119','#4363d8','#f58231',
        '#911eb4','#42d4f4','#f032e6','#bfef45','#fabed4',
        '#469990','#dcbeff','#9A6324','#ff8c00','#800000',
        '#aaffc3','#808000','#00bfff','#000075','#808080',
    ]

    def nodes_to_latlon(path_nodes, transport_type='car'):
        latlon = []
        for i in range(len(path_nodes)-1):
            u, v = path_nodes[i], path_nodes[i+1]
            geom = edge_geom_all.get((transport_type, u, v))
            if geom is None:
                cu = node_coords.get(u)
                cv = node_coords.get(v)
                if cu: latlon.append((cu[1], cu[0]))
                if cv: latlon.append((cv[1], cv[0]))
                continue
            coords = list(geom.coords)
            cu = node_coords.get(u)
            if cu and len(coords) >= 2:
                if abs(coords[-1][0] - cu[0]) < abs(coords[0][0] - cu[0]):
                    coords = coords[::-1]
            latlon.extend([(lat, lon) for lon, lat in coords])
        return latlon

    def road_segment(node_a, node_b, transport_type='car'):
        G_use = graphs.get(transport_type, graphs['car'])
        try:
            path = nx.shortest_path(G_use, source=node_a, target=node_b, weight='weight')
            return nodes_to_latlon(path, transport_type)
        except nx.NetworkXNoPath:
            ca, cb = node_coords.get(node_a), node_coords.get(node_b)
            res = []
            if ca: res.append((ca[1], ca[0]))
            if cb: res.append((cb[1], cb[0]))
            return res

    def draw_day(day_index):
        day_name = ['Maandag', 'Dinsdag', 'Woensdag', 'Donderdag', 'Vrijdag'][day_index]
        day_routes = [r for r in all_routes if r['day'] == day_index and r['client_ids']]

        center_lat = float(np.mean(node_lats_arr))
        center_lon = float(np.mean(node_lons_arr))
        m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles='CartoDB positron')

        for _, row in edges_df.iterrows():
            latlon = [(lat, lon) for lon, lat in row['geometry'].coords]
            folium.PolyLine(locations=latlon, color='#cccccc', weight=1, opacity=0.3).add_to(m)

        for r in day_routes:
            color = EMPLOYEE_COLORS[r['emp_id'] % len(EMPLOYEE_COLORS)]
            emp_name = employees_df.loc[r['emp_id'], 'name']
            transport = employees_df.loc[r['emp_id'], 'transport_type']
            nodes_seq = r['nodes']
            graph_seq = [all_nodes[n] for n in nodes_seq]
            for seg_i in range(len(graph_seq)-1):
                latlon = road_segment(graph_seq[seg_i], graph_seq[seg_i+1], transport)
                if len(latlon) >= 2:
                    folium.PolyLine(
                        locations=latlon, color=color, weight=4, opacity=0.85,
                        tooltip=f'{emp_name} | dag {day_name}'
                    ).add_to(m)

            for stop_idx, cid in enumerate(r['client_ids']):
                client = clients_df.iloc[cid]
                folium.CircleMarker(
                    location=[client['lat'], client['lon']],
                    radius=6, color='white', weight=1.5, fill=True,
                    fill_color=color, fill_opacity=0.9,
                    popup=folium.Popup(
                        f'<b>{client["name"] if "name" in client else f"Cliënt {cid}"}</b><br>'
                        f'Medewerker: {emp_name}<br>Stop {stop_idx+1}',
                        max_width=200
                    ),
                    tooltip=f'{emp_name} stop {stop_idx+1}'
                ).add_to(m)

        for emp_id, emp in employees_df.iterrows():
            color = EMPLOYEE_COLORS[emp_id % len(EMPLOYEE_COLORS)]
            has_route = any(r['emp_id'] == emp_id for r in day_routes)
            popup_text = f"{emp['name']}<br>{'Wel actief' if has_route else 'Geen bezoeken'}"
            folium.Marker(
                location=[emp['lat'], emp['lon']],
                icon=folium.DivIcon(
                    html=f'<div style="width:22px;height:22px;background:{color};border:3px solid white;border-radius:50%;box-shadow:0 2px 6px rgba(0,0,0,.5);"></div>',
                    icon_size=(22,22), icon_anchor=(11,11)
                ),
                popup=folium.Popup(popup_text, max_width=240),
                tooltip=f"{emp['name']} (thuis)"
            ).add_to(m)

        # Horizontale legenda
        legend_html = f'''
        <div style="position:fixed;bottom:20px;left:20px;z-index:1000;background:rgba(255,255,255,0.96);
                    padding:10px 14px;border-radius:8px;font-size:11px;font-family:sans-serif;
                    box-shadow:0 2px 10px rgba(0,0,0,.3);max-width:90%;overflow-x:auto;">
          <b>{day_name}</b> <span style="color:#777;">({len(day_routes)} actieve medewerkers)</span><br>
          <div style="display:flex; flex-wrap:wrap; gap:8px 16px; margin-top:6px;">'''
        for r in day_routes:
            emp_name = employees_df.loc[r['emp_id'], 'name']
            color = EMPLOYEE_COLORS[r['emp_id'] % len(EMPLOYEE_COLORS)]
            legend_html += f'<div style="display:flex; align-items:center; gap:4px;"><span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:{color};"></span><b>{emp_name}</b> <span style="color:#555;">{len(r["client_ids"])} cl.</span></div>'
        legend_html += '</div></div>'
        m.get_root().html.add_child(folium.Element(legend_html))
        return m

    # Mapping van dagindex naar Engelstalige bestandsnaam
    day_names_nl = ['Maandag', 'Dinsdag', 'Woensdag', 'Donderdag', 'Vrijdag']
    day_names_en = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday']

    # Genereer kaarten voor alle dagen
    for day in range(5):
        m = draw_day(day)
        filename = f'../output/route_{day_names_en[day]}.html'
        m.save(filename)
        print(f'Kaart voor {day_names_nl[day]} opgeslagen: {filename}')

    # Maak een overzichtspagina met tabs
    html_content = f'''<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>VRP Routes per dag</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 0;
            padding: 20px;
            background: #f5f5f5;
        }}
        .tabs {{
            display: flex;
            gap: 10px;
            margin-bottom: 20px;
            flex-wrap: wrap;
        }}
        .tab-button {{
            padding: 10px 20px;
            background: #ddd;
            border: none;
            cursor: pointer;
            font-size: 16px;
            border-radius: 5px;
            transition: 0.3s;
        }}
        .tab-button:hover {{
            background: #bbb;
        }}
        .tab-button.active {{
            background: #007bff;
            color: white;
        }}
        .map-container {{
            width: 100%;
            height: 80vh;
            border: 1px solid #ccc;
            background: white;
        }}
        iframe {{
            width: 100%;
            height: 100%;
            border: none;
        }}
    </style>
</head>
<body>
    <h1>Routes per dag - VRP planning</h1>
    <div class="tabs">
'''
    for i, name_nl in enumerate(day_names_nl):
        active = 'active' if i == 0 else ''
        html_content += f'        <button class="tab-button {active}" onclick="showDay({i})">{name_nl}</button>\n'
    html_content += '''    </div>
    <div class="map-container">
        <iframe id="mapFrame" src="route_monday.html"></iframe>
    </div>
    <script>
        const dayFiles = ['''
    for en in day_names_en:
        html_content += f"'route_{en}.html', "
    html_content = html_content.rstrip(', ') + '''];\n
        function showDay(dayIndex) {
            document.getElementById('mapFrame').src = dayFiles[dayIndex];
            var buttons = document.getElementsByClassName('tab-button');
            for (var i = 0; i < buttons.length; i++) {
                buttons[i].classList.remove('active');
            }
            buttons[dayIndex].classList.add('active');
        }
    </script>
</body>
</html>'''

    with open('../output/routes_overview.html', 'w', encoding='utf-8') as f:
        f.write(html_content)
    print('Overzichtspagina opgeslagen: ../output/routes_overview.html')

    from IPython.display import IFrame, display
    display(IFrame(src='../output/routes_overview.html', width='100%', height=700))
else:
    print('Geen routes – geen kaart beschikbaar.')

Kaart voor Maandag opgeslagen: ../output/route_monday.html
Kaart voor Dinsdag opgeslagen: ../output/route_tuesday.html
Kaart voor Woensdag opgeslagen: ../output/route_wednesday.html
Kaart voor Donderdag opgeslagen: ../output/route_thursday.html
Kaart voor Vrijdag opgeslagen: ../output/route_friday.html
Overzichtspagina opgeslagen: ../output/routes_overview.html


# 14. Dagplanning per medewerker (tekstueel)

In [38]:
if solution:
    EMPLOYEE_COLORS = [
        '#e6194b','#3cb44b','#ffe119','#4363d8','#f58231',
        '#911eb4','#42d4f4','#f032e6','#bfef45','#fabed4',
        '#469990','#dcbeff','#9A6324','#ff8c00','#800000',
        '#aaffc3','#808000','#00bfff','#000075','#808080',
    ]

    def nodes_to_latlon(path_nodes):
        latlon = []
        for i in range(len(path_nodes)-1):
            u, v = path_nodes[i], path_nodes[i+1]
            geom = edge_geom.get((u, v))
            if geom is None:
                cu = node_coords.get(u)
                cv = node_coords.get(v)
                if cu: latlon.append((cu[1], cu[0]))
                if cv: latlon.append((cv[1], cv[0]))
                continue
            coords = list(geom.coords)
            cu = node_coords.get(u)
            if cu and len(coords) >= 2:
                if abs(coords[-1][0] - cu[0]) < abs(coords[0][0] - cu[0]):
                    coords = coords[::-1]
            latlon.extend([(lat, lon) for lon, lat in coords])
        return latlon

    def road_segment(node_a, node_b, transport_type='car'):
        G_use = graphs.get(transport_type, graphs['car'])
        try:
            path = nx.shortest_path(G_use, source=node_a, target=node_b, weight='weight')
            return nodes_to_latlon(path)
        except nx.NetworkXNoPath:
            ca, cb = node_coords.get(node_a), node_coords.get(node_b)
            res = []
            if ca: res.append((ca[1], ca[0]))
            if cb: res.append((cb[1], cb[0]))
            return res

    def draw_day(day_index):
        day_name = ['Maandag', 'Dinsdag', 'Woensdag', 'Donderdag', 'Vrijdag'][day_index]
        day_routes = [r for r in all_routes if r['day'] == day_index and r['client_ids']]

        center_lat = float(np.mean(node_lats_arr))
        center_lon = float(np.mean(node_lons_arr))
        m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles='CartoDB positron')

        for _, row in edges_df.iterrows():
            latlon = [(lat, lon) for lon, lat in row['geometry'].coords]
            folium.PolyLine(locations=latlon, color='#cccccc', weight=1, opacity=0.3).add_to(m)

        for r in day_routes:
            color = EMPLOYEE_COLORS[r['emp_id'] % len(EMPLOYEE_COLORS)]
            emp_name = employees_df.loc[r['emp_id'], 'name']
            nodes_seq = r['nodes']
            graph_seq = [all_nodes[n] for n in nodes_seq]
            for seg_i in range(len(graph_seq)-1):
                transport = employees_df.loc[r['emp_id'], 'transport_type']
                latlon = road_segment(graph_seq[seg_i], graph_seq[seg_i+1], transport)
                if len(latlon) >= 2:
                    folium.PolyLine(
                        locations=latlon, color=color, weight=4, opacity=0.85,
                        tooltip=f'{emp_name} | dag {day_name}'
                    ).add_to(m)

            for stop_idx, cid in enumerate(r['client_ids']):
                client = clients_df.iloc[cid]
                folium.CircleMarker(
                    location=[client['lat'], client['lon']],
                    radius=6, color='white', weight=1.5, fill=True,
                    fill_color=color, fill_opacity=0.9,
                    popup=folium.Popup(
                        f'<b>{client["name"] if "name" in client else f"Cliënt {cid}"}</b><br>'
                        f'Medewerker: {emp_name}<br>Stop {stop_idx+1}',
                        max_width=200
                    ),
                    tooltip=f'{emp_name} stop {stop_idx+1}'
                ).add_to(m)

        for emp_id, emp in employees_df.iterrows():
            color = EMPLOYEE_COLORS[emp_id % len(EMPLOYEE_COLORS)]
            has_route = any(r['emp_id'] == emp_id for r in day_routes)
            popup_text = f"{emp['name']}<br>{'Wel actief' if has_route else 'Geen bezoeken'}"
            folium.Marker(
                location=[emp['lat'], emp['lon']],
                icon=folium.DivIcon(
                    html=f'<div style="width:22px;height:22px;background:{color};border:3px solid white;border-radius:50%;box-shadow:0 2px 6px rgba(0,0,0,.5);"></div>',
                    icon_size=(22,22), icon_anchor=(11,11)
                ),
                popup=folium.Popup(popup_text, max_width=240),
                tooltip=f"{emp['name']} (thuis)"
            ).add_to(m)

        legend_html = f'''
        <div style="position:fixed;bottom:20px;left:20px;z-index:1000;background:rgba(255,255,255,0.96);
                    padding:10px 14px;border-radius:8px;font-size:11px;font-family:sans-serif;
                    box-shadow:0 2px 10px rgba(0,0,0,.3);max-height:400px;overflow-y:auto;">
          <b>{day_name}</b><br>
          <span style="color:#777;">{len(day_routes)} actieve medewerkers</span>
          <table style="margin-top:6px;">'''
        for r in day_routes:
            emp_name = employees_df.loc[r['emp_id'], 'name']
            legend_html += f'<tr><td style="padding:2px;">•</td><td><b>{emp_name}</b></td><td style="padding-left:10px;">{len(r["client_ids"])} cliënten</td></tr>'
        legend_html += '</table></div>'
        m.get_root().html.add_child(folium.Element(legend_html))
        return m

    # Mapping van dagindex naar Engelstalige bestandsnaam (route_monday.html etc.)
    day_names_nl = ['Maandag', 'Dinsdag', 'Woensdag', 'Donderdag', 'Vrijdag']
    day_names_en = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday']
    
    # Genereer kaarten voor alle dagen
    for day in range(5):
        m = draw_day(day)
        filename = f'../output/route_{day_names_en[day]}.html'
        m.save(filename)
        print(f'Kaart voor {day_names_nl[day]} opgeslagen: {filename}')

    # Maak een overzichtspagina met tabs (verwijst naar de nieuwe bestanden)
    html_content = f'''<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>VRP Routes per dag</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 0;
            padding: 20px;
            background: #f5f5f5;
        }}
        .tabs {{
            display: flex;
            gap: 10px;
            margin-bottom: 20px;
            flex-wrap: wrap;
        }}
        .tab-button {{
            padding: 10px 20px;
            background: #ddd;
            border: none;
            cursor: pointer;
            font-size: 16px;
            border-radius: 5px;
            transition: 0.3s;
        }}
        .tab-button:hover {{
            background: #bbb;
        }}
        .tab-button.active {{
            background: #007bff;
            color: white;
        }}
        .map-container {{
            width: 100%;
            height: 80vh;
            border: 1px solid #ccc;
            background: white;
        }}
        iframe {{
            width: 100%;
            height: 100%;
            border: none;
        }}
    </style>
</head>
<body>
    <h1>Routes per dag - VRP planning</h1>
    <div class="tabs">
'''
    for i, name_nl in enumerate(day_names_nl):
        active = 'active' if i == 0 else ''
        html_content += f'        <button class="tab-button {active}" onclick="showDay({i})">{name_nl}</button>\n'
    html_content += '''    </div>
    <div class="map-container">
        <iframe id="mapFrame" src="route_monday.html"></iframe>
    </div>
    <script>
        const dayFiles = ['''
    for en in day_names_en:
        html_content += f"'route_{en}.html', "
    html_content = html_content.rstrip(', ') + '''];

        function showDay(dayIndex) {
            document.getElementById('mapFrame').src = dayFiles[dayIndex];
            var buttons = document.getElementsByClassName('tab-button');
            for (var i = 0; i < buttons.length; i++) {
                buttons[i].classList.remove('active');
            }
            buttons[dayIndex].classList.add('active');
        }
    </script>
</body>
</html>'''

    with open('../output/routes_overview.html', 'w', encoding='utf-8') as f:
        f.write(html_content)
    print('Overzichtspagina opgeslagen: ../output/routes_overview.html')

    from IPython.display import IFrame, display
    display(IFrame(src='../output/routes_overview.html', width='100%', height=700))
else:
    print('Geen routes – geen kaart beschikbaar.')

NameError: name 'edge_geom' is not defined